# FaceNet vs DeepFace — Ewaluacja na LFW Dataset
Porównanie modeli rozpoznawania twarzy: FaceNet, Facenet512, ArcFace (przez bibliotekę DeepFace)
Dataset: Labeled Faces in the Wild (LFW) — standardowy benchmark weryfikacji twarzy

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jessicali9530/lfw-dataset")

print("Path to dataset files:", path)

100%|██████████| 112M/112M [00:04<00:00, 27.9MB/s] 

Extracting files...


Path to dataset files: C:\Users\Dawid\.cache\kagglehub\datasets\jessicali9530\lfw-dataset\versions\4


## Imports and hardware test

In [4]:
import os
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

import torch
from deepface import DeepFace

print(f"GPU ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


GPU ready: True
GPU: NVIDIA GeForce RTX 5070 Ti


In [34]:
img1 = "C:\\Users\\Dawid\\Desktop\\praca inżynierska\\real-time-alaysis-research\\datasets\\lfw_dataset\\lfw-deepfunneled\\lfw-deepfunneled\\Abdoulaye_Wade\\Abdoulaye_Wade_0001.jpg"
img2 = "C:\\Users\\Dawid\\Desktop\\praca inżynierska\\real-time-alaysis-research\\datasets\\lfw_dataset\\lfw-deepfunneled\\lfw-deepfunneled\\Abdoulaye_Wade\\Abdoulaye_Wade_0002.jpg"
print(os.path.isfile(img1))
print(os.path.isfile(img2))
result = DeepFace.verify(img1_path=img1,
                        img2_path=img2,
                        model_name='ArcFace',
                        enforce_detection=False,
                        detector_backend='retinaface')
print("Is same person:", result['verified'])

True
True


ValueError: Exception while processing img1_path

## Dataset path and configuration

In [5]:
DATASET_PATH = os.path.abspath("../../datasets/lfw_dataset")
LFW_DIR      = os.path.join(DATASET_PATH, "lfw-deepfunneled", "lfw-deepfunneled")
OUTPUT_DIR   = "runs/facenet_deepface"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset path: {DATASET_PATH}")
print(f"LFW images:   {LFW_DIR}")
print(f"Output:       {OUTPUT_DIR}")

Dataset path: c:\Users\Dawid\Desktop\praca inżynierska\real-time-alaysis-research\datasets\lfw_dataset
LFW images:   c:\Users\Dawid\Desktop\praca inżynierska\real-time-alaysis-research\datasets\lfw_dataset\lfw-deepfunneled\lfw-deepfunneled
Output:       runs/facenet_deepface


## Global configuration

In [7]:
# ── Liczba par do testu ────────────────────────────────────────────────────────
# LFW standard: 300 par pozytywnych + 300 par negatywnych = 600 par
# Możesz zwiększyć do 1000 dla dokładniejszych wyników (dłuższy czas)
NUM_POSITIVE_PAIRS = 300   # pary tej samej osoby
NUM_NEGATIVE_PAIRS = 300   # pary różnych osób
SEED               = 42

# ── Modele do porównania ───────────────────────────────────────────────────────
# FaceNet    — oryginalny model Google (128-wymiarowe embeddingi)
# Facenet512 — rozszerzona wersja FaceNet (512-wymiarowe embeddingi, dokładniejsza)
# ArcFace    — state-of-the-art model (dla porównania z FaceNet)
MODELS = [
    "Facenet",
    "Facenet512",
    "ArcFace",
]

# ── Metryka odległości ─────────────────────────────────────────────────────────
DISTANCE_METRIC = "cosine"  # cosine / euclidean / euclidean_l2

# ── Detektor twarzy ───────────────────────────────────────────────────────────
# retinaface — najdokładniejszy, wolniejszy
# opencv     — szybki, mniej dokładny
# skip       — pomiń detekcję (LFW jest pre-aligned więc to wystarczy)
DETECTOR = "skip"  # LFW deepfunneled jest już wyrównany — skip jest OK

print(f"Modele:         {MODELS}")
print(f"Par pozytywnych: {NUM_POSITIVE_PAIRS}")
print(f"Par negatywnych: {NUM_NEGATIVE_PAIRS}")
print(f"Metryka:        {DISTANCE_METRIC}")
print(f"Detektor:       {DETECTOR}")

Modele:         ['Facenet', 'Facenet512', 'ArcFace']
Par pozytywnych: 300
Par negatywnych: 300
Metryka:        cosine
Detektor:       skip


## Build test pairs from LFW dataset
Tworzymy pary (same_person=True/False) na podstawie struktury folderów LFW.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

lfw_path = Path(LFW_DIR)

# Wczytaj wszystkie osoby i ich zdjęcia
persons = {}
for person_dir in sorted(lfw_path.iterdir()):
    if not person_dir.is_dir():
        continue
    images = sorted(list(person_dir.glob("*.jpg")))
    if len(images) > 0:
        persons[person_dir.name] = images

print(f"Liczba osób w LFW:   {len(persons)}")
print(f"Łączna liczba zdjęć: {sum(len(v) for v in persons.values())}")

# Osoby z ≥2 zdjęciami — do par pozytywnych
multi_persons = {k: v for k, v in persons.items() if len(v) >= 2}
print(f"Osoby z ≥2 zdjęciami: {len(multi_persons)}")

# ── Pary pozytywne (ta sama osoba, różne zdjęcia) ─────────────────────────────
positive_pairs = []
multi_list = list(multi_persons.items())
random.shuffle(multi_list)

for name, images in multi_list:
    if len(positive_pairs) >= NUM_POSITIVE_PAIRS:
        break
    imgs = random.sample(images, 2)
    positive_pairs.append({
        "img1":        str(imgs[0]),
        "img2":        str(imgs[1]),
        "same_person": True,
        "person":      name
    })

# ── Pary negatywne (różne osoby) ──────────────────────────────────────────────
negative_pairs = []
all_persons_list = list(hile len(negative_pairs) < NUM_NEGATIVE_PAIRS:
    (name1, imgs1), (name2, imgs2) = random.sample(all_persons_list, 2)
    if name1 == name2:
        continue
    negative_pairs.append({
        "img1":        str(random.choice(imgs1)),
        "img2":        str(random.choice(imgs2)),
        "same_person": False,
        "person":      f"{name1} vs {name2}"
    })

# Połącz i potasuj
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)
df_pairs = pd.DataFrame(all_pairs)

print(f"\nPary pozytywne: {len(positive_pairs)}")
print(f"Pary negatywne: {len(negative_pairs)}")
print(f"Łącznie par:    {len(df_pairs)}")
df_pairs.head()

Liczba osób w LFW:   5749
Łączna liczba zdjęć: 13233
Osoby z ≥2 zdjęciami: 1680

Pary pozytywne: 300
Pary negatywne: 300
Łącznie par:    600


,img1,img2,same_person,person
0,c:\Users\Dawid\Desktop\praca inżynierska\real-...,c:\Users\Dawid\Desktop\praca inżynierska\real-...,False,Ali_Fallahian vs Lee_Baca
1,c:\Users\Dawid\Desktop\praca inżynierska\real-...,c:\Users\Dawid\Desktop\praca inżynierska\real-...,True,Inocencio_Arias
2,c:\Users\Dawid\Desktop\praca inżynierska\real-...,c:\Users\Dawid\Desktop\praca inżynierska\real-...,False,Bob_Eskridge vs Elizabeth_Dole
3,c:\Users\Dawid\Desktop\praca inżynierska\real-...,c:\Users\Dawid\Desktop\praca inżynierska\real-...,True,Priscilla_Presley
4,c:\Users\Dawid\Desktop\praca inżynierska\real-...,c:\Users\Dawid\Desktop\praca inżynierska\real-...,False,Paris_Hilton vs Fabian_Vargas


## Evaluation — verify all pairs for each model
Główna pętla ewaluacji — dla każdego modelu sprawdzamy każdą parę.

In [25]:
from PIL import Image
results_all = {}  # wyniki dla każdego modelu

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"  Model: {model_name}")
    print(f"{'='*60}")

    distances   = []
    predictions = []
    labels      = []
    times       = []
    errors      = 0

    for _, row in tqdm(df_pairs.iterrows(), total=len(df_pairs), desc=model_name):
        try:
            img1 = row.img1
            Image.open(img1).convert("RGB")
            #DeepFace.detectFace(img1, detector_backend=DETECTOR)
            img2 = row.img2
            Image.open(img2).convert("RGB")
            #DeepFace.detectFace(img2, detector_backend=DETECTOR)

            if not os.path.isfile(img1) or not os.path.isfile(img2):
                print(f"  WARNING: plik nie istnieje: {img1 if not os.path.isfile(img1) else img2}")
                errors += 1
                continue

            t0 = time.perf_counter()
            result = DeepFace.verify(
                img1_path        = img1,
                img2_path        = img2,
                model_name       = model_name,
                distance_metric  = DISTANCE_METRIC,
                detector_backend = DETECTOR,
                enforce_detection= False,  # nie przerywaj gdy brak twarzy
                align            = True
            )
            elapsed = time.perf_counter() - t0

            distances.append(result["distance"])
            predictions.append(1 if result["verified"] else 0)
            labels.append(1 if row["same_person"] else 0)
            times.append(elapsed)

        except Exception as e:
            print(f"  ERROR: model {model_name} nie przetworzył pary: {img1} , {img2}; wyjątek: {type(e).__name__}: {e}")
            errors += 1
            continue

    labels      = np.array(labels)
    predictions = np.array(predictions)
    distances   = np.array(distances)

    # ── Metryki ────────────────────────────────────────────────────────────────
    accuracy = accuracy_score(labels, predictions)
    auc      = roc_auc_score(labels, -distances)  # niższa odległość = bardziej podobne
    cm       = confusion_matrix(labels, predictions)

    # TN, FP, FN, TP
    tn, fp, fn, tp = cm.ravel()
    far = fp / (fp + tn + 1e-8)  # False Acceptance Rate — obcy akceptowany jako znajomy
    frr = fn / (fn + tp + 1e-8)  # False Rejection Rate — znajomy odrzucony

    avg_time_ms = np.mean(times) * 1000
    fps         = 1.0 / np.mean(times)

    results_all[model_name] = {
        "accuracy":   round(accuracy, 4),
        "auc":        round(auc, 4),
        "far":        round(far, 4),
        "frr":        round(frr, 4),
        "tp":         int(tp),
        "tn":         int(tn),
        "fp":         int(fp),
        "fn":         int(fn),
        "avg_ms":     round(avg_time_ms, 2),
        "fps":        round(fps, 2),
        "errors":     errors,
        "distances":  distances,
        "labels":     labels,
        "predictions":predictions
    }

    print(f"\n  Accuracy:  {accuracy:.4f}")
    print(f"  AUC:       {auc:.4f}")
    print(f"  FAR:       {far:.4f}  (fałszywe akceptacje)")
    print(f"  FRR:       {frr:.4f}  (fałszywe odrzucenia)")
    print(f"  Avg time:  {avg_time_ms:.1f} ms/para")
    print(f"  Errors:    {errors} (zdjęcia bez twarzy)")


  Model: Facenet


Facenet:   0%|          | 0/600 [00:25<?, ?it/s]


KeyboardInterrupt: 

## Results summary table

In [ ]:
rows = []
for model_name, r in results_all.items():
    rows.append({
        "Model":    model_name,
        "Accuracy": r["accuracy"],
        "AUC":      r["auc"],
        "FAR":      r["far"],
        "FRR":      r["frr"],
        "TP":       r["tp"],
        "TN":       r["tn"],
        "FP":       r["fp"],
        "FN":       r["fn"],
        "Avg_ms":   r["avg_ms"],
        "FPS":      r["fps"]
    })

df_results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)

print("\n" + "="*80)
print("  PORÓWNANIE MODELI — FaceNet vs DeepFace (LFW)")
print("="*80)
print(df_results.to_string(index=False))

# Zapisz do CSV
csv_path = os.path.join(OUTPUT_DIR, "results_summary.csv")
df_results.to_csv(csv_path, index=False)
print(f"\nZapisano: {csv_path}")

## Plots — ROC curves and distance distributions

In [ ]:
COLORS = {"Facenet": "#378ADD", "Facenet512": "#0F6E56", "ArcFace": "#534AB7"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Wykres 1: Krzywe ROC ───────────────────────────────────────────────────────
ax = axes[0]
for model_name, r in results_all.items():
    fpr, tpr, _ = roc_curve(r["labels"], -r["distances"])
    ax.plot(fpr, tpr,
            label=f"{model_name} (AUC={r['auc']:.3f})",
            color=COLORS.get(model_name, "gray"),
            linewidth=2)

ax.plot([0,1],[0,1], "k--", alpha=0.4, linewidth=1)
ax.set_xlabel("False Positive Rate (FAR)", fontsize=12)
ax.set_ylabel("True Positive Rate (1-FRR)", fontsize=12)
ax.set_title("Krzywe ROC — weryfikacja twarzy (LFW)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Wykres 2: Rozkład odległości ───────────────────────────────────────────────
ax = axes[1]
for model_name, r in results_all.items():
    color = COLORS.get(model_name, "gray")
    same_dist = r["distances"][r["labels"] == 1]
    diff_dist = r["distances"][r["labels"] == 0]
    ax.hist(same_dist, bins=30, alpha=0.4, color=color,
            label=f"{model_name} — ta sama osoba")
    ax.hist(diff_dist, bins=30, alpha=0.2, color=color,
            label=f"{model_name} — różne osoby", linestyle="dashed",
            histtype="step", linewidth=2)

ax.set_xlabel(f"Odległość cosine", fontsize=12)
ax.set_ylabel("Liczba par", fontsize=12)
ax.set_title("Rozkład odległości embeddingów", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "roc_and_distances.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Zapisano: {plot_path}")

## Accuracy vs Speed plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model_name, r in results_all.items():
    color = COLORS.get(model_name, "gray")
    ax.scatter(r["avg_ms"], r["accuracy"],
               color=color, s=200, zorder=5,
               edgecolors="white", linewidth=1.5)
    ax.annotate(model_name,
                (r["avg_ms"], r["accuracy"]),
                textcoords="offset points",
                xytext=(8, 4), fontsize=10,
                color=color, fontweight="bold")

ax.set_xlabel("Średni czas weryfikacji [ms] (im mniej tym lepiej →)", fontsize=12)
ax.set_ylabel("Accuracy (im wyżej tym lepiej ↑)", fontsize=12)
ax.set_title("Kompromis: dokładność vs szybkość", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path2 = os.path.join(OUTPUT_DIR, "accuracy_vs_speed.png")
plt.savefig(plot_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Zapisano: {plot_path2}")

## FPS measurement — CPU and GPU
Pomiar szybkości ekstrakcji embeddingów (bez wykrywania twarzy).

In [ ]:
# Weź jeden przykładowy obraz do benchmarku
sample_img = df_pairs.iloc[0]["img1"]
FPS_RUNS   = 20

print("Pomiar FPS — ekstrakcja embeddingów")
print("-" * 50)

for model_name in MODELS:
    # Rozgrzewka
    for _ in range(3):
        DeepFace.represent(
            img_path         = sample_img,
            model_name       = model_name,
            detector_backend = DETECTOR,
            enforce_detection= False
        )

    # Pomiar
    times = []
    for _ in range(FPS_RUNS):
        t0 = time.perf_counter()
        DeepFace.represent(
            img_path         = sample_img,
            model_name       = model_name,
            detector_backend = DETECTOR,
            enforce_detection= False
        )
        times.append(time.perf_counter() - t0)

    avg_ms = np.mean(times) * 1000
    fps    = 1.0 / np.mean(times)
    print(f"  {model_name:<15} {avg_ms:>8.1f} ms   {fps:>8.1f} FPS")

## Final summary — ready for comparison table

In [ ]:
print("\n" + "="*70)
print("  WYNIKI DO TABELI ZBIORCZEJ")
print("="*70)
print(f"Dataset: LFW deepfunneled")
print(f"Pary:    {NUM_POSITIVE_PAIRS} pozytywne + {NUM_NEGATIVE_PAIRS} negatywne")
print(f"Metryka: {DISTANCE_METRIC}")
print(f"Detektor:{DETECTOR}")
print("-"*70)
print(f"{'Model':<15} {'Accuracy':>10} {'AUC':>8} {'FAR':>8} {'FRR':>8} {'ms/para':>10}")
print("-"*70)

for model_name in MODELS:
    r = results_all[model_name]
    print(f"{model_name:<15} {r['accuracy']:>10.4f} {r['auc']:>8.4f} "
          f"{r['far']:>8.4f} {r['frr']:>8.4f} {r['avg_ms']:>10.1f}")

print("-"*70)
print("\nObjaśnienia metryk:")
print("  Accuracy — odsetek poprawnie sklasyfikowanych par")
print("  AUC      — pole pod krzywą ROC (1.0 = idealny)")
print("  FAR      — False Acceptance Rate: obcy zaakceptowany jako znajomy")
print("  FRR      — False Rejection Rate: znajomy odrzucony")
print("  ms/para  — średni czas weryfikacji jednej pary zdjęć")